# Understanding Code Ownership in siege_utilities

**Scenario:** A new team member joins and needs to understand the repo:
which modules change most, what the commit patterns look like, and who
owns what. Rather than reading every file, they use the git/ module to
query the repository's own history programmatically.

## What this shows

Analyze repository status, commit history, branch ahead/behind state, and branch inventory against a temporary local fixture repository. This is a pure/offline notebook: it should run without network, credentials, GDAL, Spark, or touching the user's real repository state.


## 1. Build a Temporary Fixture Repository


In [ ]:
import subprocess
import tempfile
from pathlib import Path

from siege_utilities.git.git_status import get_repository_status, get_branch_info

fixture_dir = tempfile.TemporaryDirectory()
fixture_root = Path(fixture_dir.name)
repo_root = fixture_root / "demo-repo"
remote_root = fixture_root / "demo-remote.git"
repo_root.mkdir()

def git(*args, cwd=repo_root):
    return subprocess.run(
        ["git", *args],
        cwd=cwd,
        check=True,
        capture_output=True,
        text=True,
        timeout=30,
    )

git("init")
git("config", "user.email", "notebook@example.test")
git("config", "user.name", "Notebook Demo")
git("init", "--bare", str(remote_root), cwd=fixture_root)
git("remote", "add", "origin", str(remote_root))
(repo_root / "README.md").write_text("# Demo repo\n")
git("add", "README.md")
git("commit", "-m", "docs: seed demo repo")
git("branch", "-M", "main")
git("push", "-u", "origin", "main")
(repo_root / "analysis.py").write_text("print('hello')\n")
git("add", "analysis.py")
git("commit", "-m", "feat: add analysis script")
git("checkout", "-b", "demo/change")
(repo_root / "analysis.py").write_text("print('hello from branch')\n")
git("commit", "-am", "fix: update branch output")

status = get_repository_status(repo_path=str(repo_root))
print(f"Fixture repo: {repo_root}")
print(f"Current branch: {status['current_branch']}")
print(f"Clean: {status['working_directory_clean']}")


## 2. Recent Commit History and Categorization

The branch analyzer parses conventional commits (feat, fix, docs, etc.)
and categorizes them. This tells you what kind of work is active.

In [ ]:
from siege_utilities.git.branch_analyzer import get_commit_history, categorize_commits

commits = get_commit_history(limit=20, repo_path=str(repo_root))
categories = categorize_commits(commits)

print(f"Last {len(commits)} fixture commits by category:")
print(f"{'Category':<15} {'Count':>5}")
print("-" * 22)
for cat, items in sorted(categories.items(), key=lambda x: -len(x[1])):
    if items:
        print(f"{cat:<15} {len(items):>5}")


## 3. Branch Status: How Far Ahead/Behind?

`analyze_branch_status` shows the relationship between the current
branch and main — useful for knowing if you need to rebase.

In [ ]:
from siege_utilities.git.branch_analyzer import analyze_branch_status

branch_status = analyze_branch_status(repo_path=str(repo_root))

print("Branch analysis:")
for key, value in branch_status.items():
    print(f"  {key}: {value}")


## 4. Branch Inventory

See all local and remote branches with their latest activity.
Stale branches with no recent commits are cleanup candidates.

In [ ]:
branch_info = get_branch_info(repo_path=str(repo_root))

local_branches = branch_info.get("local_branches", [])
remote_branches = branch_info.get("remote_branches", [])
current_branch = branch_info.get("current_branch")

def branch_name(branch):
    return branch.get("name", "unknown") if isinstance(branch, dict) else str(branch)

print(f"Local branches: {branch_info.get('total_local_branches', len(local_branches))}")
for branch in local_branches:
    name = branch_name(branch)
    current = " <-" if name == current_branch else ""
    print(f"  {name}{current}")

print(f"\nRemote branches: {branch_info.get('total_remote_branches', len(remote_branches))}")
for branch in remote_branches:
    print(f"  {branch_name(branch)}")


## Related

- Source: `siege_utilities/git/git_status.py`, `siege_utilities/git/branch_analyzer.py`
- Tests: `tests/test_git_status.py`, `tests/test_git_branch_analyzer_errors.py`, `tests/test_git_operations.py`
- Notebook governance: `tests/test_notebook_hygiene.py`, `tests/test_notebooks.py`, `scripts/check_notebook_inventory.py`
